In [5]:
import pandas as pd
import numpy as np
import os

# 1. Load the raw data with correct encoding
listings = pd.read_csv("../data/raw/Listings.csv", encoding='latin1', low_memory=False)
reviews = pd.read_csv("../data/raw/Reviews.csv", encoding='latin1', low_memory=False)

# 2. Filter using the exact Maven Analytics columns
listings_keep = [
    "listing_id", "city", "room_type", "price", "minimum_nights",
    "latitude", "longitude", "review_scores_rating"
]
listings = listings[listings_keep].copy()

# 3. Handle data clean-up and numeric formatting
listings["price"] = pd.to_numeric(listings["price"], errors='coerce')

listings.drop_duplicates(inplace=True)
listings.dropna(subset=["price", "city", "room_type"], inplace=True)
listings["review_scores_rating"] = listings["review_scores_rating"].fillna(0)

# 4. Filter pricing outliers and unrealistic minimum nights
listings = listings[listings["price"] > 0]
listings = listings[listings["price"] < listings["price"].quantile(0.99)]
listings = listings[listings["minimum_nights"] <= 365]
print("Listings shape after cleaning:", listings.shape)

# 5. Clean the massive reviews dataset (using actual columns: listing_id, review_id, date, reviewer_id)
reviews.drop_duplicates(inplace=True)
reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")
reviews.dropna(subset=["date"], inplace=True)
print("Reviews shape after cleaning:", reviews.shape)

# 6. Save clean versions out to your pipeline storage
os.makedirs("../data/cleaned", exist_ok=True)
listings.to_csv("../data/cleaned/listings_clean.csv", index=False)
reviews.to_csv("../data/cleaned/reviews_clean.csv", index=False)
print("Saved cleaned files successfully!")

print("\nFinal missing value check for listings:\n", listings.isnull().sum())

Listings shape after cleaning: (276693, 8)
Reviews shape after cleaning: (5373143, 4)
Saved cleaned files successfully!

Final missing value check for listings:
 listing_id              0
city                    0
room_type               0
price                   0
minimum_nights          0
latitude                0
longitude               0
review_scores_rating    0
dtype: int64
